In [ ]:
import json
import os

import mlflow
from datasets import load_dataset
from dotenv import load_dotenv
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from mlflow.genai.datasets import create_dataset, delete_dataset
from mlflow.genai.scorers.phoenix import Toxicity

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "dataset_tracking_example"

In [4]:
# Warning - This step may take a while to download the dataset - The dataset size is ~40GB
wikipedia_dataset = load_dataset("wikimedia/wikipedia", "20231101.en")
wikipedia_dataset

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 6407814
    })
})

In [5]:
KEYWORD = "artificial intelligence"


# Filter for articles where the specific keyword if it appears at least 3 times
def filter_article_by_keyword(article):
    KEYWORD = "artificial intelligence"
    text_lower = article["text"].lower()
    count = text_lower.count(KEYWORD)
    return count >= 3


filtered_articles_dataset = wikipedia_dataset["train"].filter(
    filter_article_by_keyword,
    desc="Filtering articles with at least 3 occurrences of the keyword",
    batch_size=1000,
    writer_batch_size=1000,
    num_proc=4,
)

filtered_articles_dataset

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1683
})

In [ ]:
filtered_articles_df = filtered_articles_dataset.to_pandas()
filtered_articles_id_title_json = filtered_articles_df[["id", "title"]].to_json(
    orient="records"
)
MODEL = "openai"

if MODEL == "google":
    rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
        check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
        max_bucket_size=10,  # Controls the maximum burst size.
    )

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        rate_limiter=rate_limiter,
    )
elif MODEL == "openai":
    llm = ChatOpenAI(
        model="gpt-5-nano",
    )
else:
    raise ValueError(f"Unsupported MODEL: {MODEL}")

prompt = f"""Your are provided with the id and title of the wikipedia articles. Identify the top 100 articles which matches best to the Topic - {KEYWORD} and return the ids of the articles in a list format.
Make sure to return only 100 ids in a json list format.
{filtered_articles_id_title_json}
"""

predicted_ids = llm.invoke(prompt)
predicted_ids


AIMessage(content='["713","1164","1208","2142","2846","2862","4715","5323","5561","5626","5703","4075738","4103607","4168072","4244428","4244882","4253446","4325491","4357120","4384927","4395638","4420730","4513331","4522868","4559681","4565055","4593326","4678739","4733844","4843650","4848621","4852787","4920830","4986640","5033373","5065056","5069239","5071866","5175523","5192353","5254736","5276122","5356861","5362865","5368665","5397677","5415830","5439872","5589387","5649586","5725671","5732209","5837233","5841092","5843242","5906926","6112811","6320384","6338699","6476399","6477222","6501490","6585513","6610123","6691035","7092991","7093162","7101688","71016058","71106184","71116611","71151961","71642695","71664574","71692942","71700540","71751381","71851483","71974276","72093304","72417803","72514599","73291899","73312151","73363598","73400769","73455792","73645807","73682940","73849538","73897657","73930043","74045044","74091926","74185828","74209694","74209862","74226922","742

In [9]:
predicted_ids_list = json.loads(
    predicted_ids.content.replace("```json", "").replace("```", "")
)
len(predicted_ids_list)

100

In [10]:
filtered_articles_df = filtered_articles_df[
    filtered_articles_df["id"].isin(predicted_ids_list)
]
print(filtered_articles_df.shape)
filtered_articles_df.head(10)

(100, 4)


,id,url,title,text
0,713,https://en.wikipedia.org/wiki/Android%20%28rob...,Android (robot),An android is a humanoid robot or other artifi...
1,1164,https://en.wikipedia.org/wiki/Artificial%20int...,Artificial intelligence,Artificial intelligence (AI) is the intelligen...
2,1208,https://en.wikipedia.org/wiki/Alan%20Turing,Alan Turing,Alan Mathison Turing (; 23 June 1912 – 7 June...
4,2142,https://en.wikipedia.org/wiki/List%20of%20arti...,List of artificial intelligence projects,"The following is a list of current and past, n..."
5,2846,https://en.wikipedia.org/wiki/Ai,Ai,"AI is artificial intelligence, intellectual ab..."
6,2862,https://en.wikipedia.org/wiki/AI-complete,AI-complete,"In the field of artificial intelligence, the m..."
7,4715,https://en.wikipedia.org/wiki/Boolean%20satisf...,Boolean satisfiability problem,"In logic and computer science, the Boolean sat..."
8,5323,https://en.wikipedia.org/wiki/Computer%20science,Computer science,"Computer science is the study of computation, ..."
9,5561,https://en.wikipedia.org/wiki/Computational%20...,Computational linguistics,Computational linguistics is an interdisciplin...
10,5626,https://en.wikipedia.org/wiki/Cognitive%20science,Cognitive science,"Cognitive science is the interdisciplinary, sc..."


In [11]:
dataset = mlflow.data.from_pandas(
    df=filtered_articles_df, name=f"top_100_articles_{KEYWORD.replace(' ', '_')}"
)

with mlflow.start_run():
    mlflow.log_input(dataset, context="raw_data")

🏃 View run dazzling-kit-348 at: http://localhost:5000/#/experiments/1/runs/21ba51fbfd1145be93bffe7711d1cb07
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [12]:
data_list = []

for record in filtered_articles_df["text"].tolist():
    data_list.append({"inputs": {"text": record}, "expectations": {}})
data_list[0]

{'inputs': {'text': 'An android is a humanoid robot or other artificial being often made from a flesh-like material. Historically, androids were completely within the domain of science fiction and frequently seen in film and television, but advances in robot technology now allow the design of functional and realistic humanoid robots.\n\nTerminology\n\nThe Oxford English Dictionary traces the earliest use (as "Androides") to Ephraim Chambers\' 1728 Cyclopaedia, in reference to an automaton that St. Albertus Magnus allegedly created. By the late 1700s, "androides", elaborate mechanical devices resembling humans performing human activities, were displayed in exhibit halls.\nThe term "android" appears in US patents as early as 1863 in reference to miniature human-like toy automatons. The term android was used in a more modern sense by the French author Auguste Villiers de l\'Isle-Adam in his work Tomorrow\'s Eve (1886). This story features an artificial humanlike robot named Hadaly. As sai

In [13]:
summary_dataset = create_dataset(
    name="summarization_dataset",
    tags={"type": "summary", "source": "wikipedia"},
)

In [14]:
summary_dataset.merge_records(data_list[:10])

In [ ]:
# from mlflow.genai.datasets import get_dataset
# get_dataset(dataset_id=summary_dataset.dataset_id)

In [15]:
def predict_fn(text) -> str:
    prompt = f"""Summarize the text below as a bullet point list of the most important points.
    Text: {text}"""
    response = llm.invoke(prompt)
    return response.content


# 3.Run the evaluation
results = mlflow.genai.evaluate(
    data=summary_dataset, predict_fn=predict_fn, scorers=[Toxicity(model="openai:/gpt-5-mini"),],
)

2026/02/05 20:51:51 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?] 

In [16]:
results.result_df

,trace_id,Toxicity/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-44c5380362501e3686051b0ee674af9f,non-toxic,"{""info"": {""trace_id"": ""tr-44c5380362501e368605...",None,OK,1770304934569,24254,"{'text': 'In logic and computer science, the B...","- SAT: given a Boolean formula, decide if ther...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': '1666b064-c3fd-4caa-...,"[{'trace_id': 'RMU4A2JQHjaGBRsO5nSvnw==', 'spa...",[{'assessment_id': 'a-1522387cf09a493e84553dfc...
1,tr-fedb3e4fae16bf0d3271875ba0ce8d90,non-toxic,"{""info"": {""trace_id"": ""tr-fedb3e4fae16bf0d3271...",None,OK,1770304934569,20575,{'text': 'An android is a humanoid robot or ot...,- An android is a humanoid robot or artificial...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': '98f575a1-09d0-4ff3-...,"[{'trace_id': '/ts+T64Wvw0ycYdboM6NkA==', 'spa...",[{'assessment_id': 'a-ee7ad9fad0164b168dd71064...
2,tr-489163cefcf211af37685e966be4a2f3,non-toxic,"{""info"": {""trace_id"": ""tr-489163cefcf211af3768...",None,OK,1770304934569,43259,{'text': 'The following is a list of current a...,"- Overview: A list of current and past, non-cl...","{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': '2c6eaab5-8757-43a6-...,"[{'trace_id': 'SJFjzvzyEa83aF6Wa+Si8w==', 'spa...",[{'assessment_id': 'a-af736202b1c44878bc4b62f6...
3,tr-45b4d106a45817976f04b6771cc42620,non-toxic,"{""info"": {""trace_id"": ""tr-45b4d106a45817976f04...",None,OK,1770304934577,21149,{'text': 'Alan Mathison Turing (; 23 June 191...,- Alan Turing (1912–1954) was an English mathe...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': 'ee53a2b2-19f0-4899-...,"[{'trace_id': 'RbTRBqRYF5dvBLZ3HMQmIA==', 'spa...",[{'assessment_id': 'a-33b54fb6d9104862b350fd74...
4,tr-3f806aaaaa8666e03388c7cd4653fab0,non-toxic,"{""info"": {""trace_id"": ""tr-3f806aaaaa8666e03388...",None,OK,1770304934583,16345,{'text': 'In the field of artificial intellige...,- AI-complete (AI-hard) problems are as diffic...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': '4eb7592f-89d6-40a8-...,"[{'trace_id': 'P4BqqqqGZuAziMfNRlP6sA==', 'spa...",[{'assessment_id': 'a-ee941590922e4c43b315c79b...
5,tr-5b79d3636b203bafad7cc9b101f7d301,non-toxic,"{""info"": {""trace_id"": ""tr-5b79d3636b203bafad7c...",None,OK,1770304934586,30540,"{'text': 'AI is artificial intelligence, intel...",- AI usually stands for artificial intelligenc...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': 'de2a412e-b44a-46f7-...,"[{'trace_id': 'W3nTY2sgO6+tfMmxAffTAQ==', 'spa...",[{'assessment_id': 'a-8adc00547e784d8ba9f3e94c...
6,tr-a81cff2df1eaf5b1a691c403fa581192,non-toxic,"{""info"": {""trace_id"": ""tr-a81cff2df1eaf5b1a691...",None,OK,1770304934588,23821,{'text': 'Cognitive science is the interdiscip...,- Cognitive science is an interdisciplinary st...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': '0d36e865-b00f-4c28-...,"[{'trace_id': 'qBz/LfHq9bGmkcQD+lgRkg==', 'spa...",[{'assessment_id': 'a-28a022ccd1e84a73a8a7db16...
7,tr-f7b59566dfc4a22d1e31aa355482b816,non-toxic,"{""info"": {""trace_id"": ""tr-f7b59566dfc4a22d1e31...",None,OK,1770304934591,21401,{'text': 'Computer science is the study of com...,- Computer science is the study of computation...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': '1dd3722a-be2f-4ac4-...,"[{'trace_id': '97WVZt/Eoi0eMao1VIK4Fg==', 'spa...",[{'assessment_id': 'a-07a0ca4521494b77babbed2f...
8,tr-50edf02811d90c767ce875885aaf4e05,non-toxic,"{""info"": {""trace_id"": ""tr-50edf02811d90c767ce8...",None,OK,1770304934594,14207,{'text': 'Computational linguistics is an inte...,- Computational linguistics is an interdiscipl...,"{'mlflow.trace_schema.version': '3', 'mlflow.s...",{'mlflow.eval.requestId': 'e43ce23f-a32f-4f36-...,"[{'trace_id': 'UO3w

In [17]:
annotated_dataset = create_dataset(
    name="annotated_dataset",
    tags={"type": "annotated", "source": "wikipedia"},
)

In [18]:
annotated_data_list = []
for index, row in results.result_df[["request", "response"]].iterrows():
    annotated_data_list.append(
        {
            "inputs": {"text": row["request"]["text"]},
            "expectations": {"response": row["response"]},
        }
    )

annotated_data_list[0]

{'inputs': {'text': 'In logic and computer science, the Boolean satisfiability problem (sometimes called propositional satisfiability problem and abbreviated SATISFIABILITY, SAT or B-SAT) is the problem of determining if there exists an interpretation that satisfies a given Boolean formula. In other words, it asks whether the variables of a given Boolean formula can be consistently replaced by the values TRUE or FALSE in such a way that the formula evaluates to TRUE. If this is the case, the formula is called satisfiable. On the other hand, if no such assignment exists, the function expressed by the formula is FALSE for all possible variable assignments and the formula is unsatisfiable. For example, the formula "a AND NOT b" is satisfiable because one can find the values a\xa0=\xa0TRUE and b\xa0=\xa0FALSE, which make (a AND NOT b)\xa0=\xa0TRUE. In contrast, "a AND NOT a" is unsatisfiable.\n\nSAT is the first problem that was proven to be NP-complete; see Cook–Levin theorem. This means 

In [19]:
annotated_dataset.merge_records(annotated_data_list)

In [20]:
delete_dataset(dataset_id=annotated_dataset.dataset_id)

In [21]:
traces = mlflow.search_traces(max_results=10, return_type="list", run_id=results.run_id)
traces

[Trace(trace_id=tr-50edf02811d90c767ce875885aaf4e05),
 Trace(trace_id=tr-ac3be6fbf20b3256630dcdcb3e23d224),
 Trace(trace_id=tr-f7b59566dfc4a22d1e31aa355482b816),
 Trace(trace_id=tr-a81cff2df1eaf5b1a691c403fa581192),
 Trace(trace_id=tr-5b79d3636b203bafad7cc9b101f7d301),
 Trace(trace_id=tr-3f806aaaaa8666e03388c7cd4653fab0),
 Trace(trace_id=tr-45b4d106a45817976f04b6771cc42620),
 Trace(trace_id=tr-44c5380362501e3686051b0ee674af9f),
 Trace(trace_id=tr-489163cefcf211af37685e966be4a2f3),
 Trace(trace_id=tr-fedb3e4fae16bf0d3271875ba0ce8d90)]

In [22]:
annotated_data_list = []
for trace in traces:
    trace_id = trace.to_dict()["info"]["trace_id"]
    request = trace.data._get_root_span().inputs["text"]
    response = trace.data._get_root_span().outputs
    annotated_data_list.append(
        {"inputs": {"text": request}, "expectations": {"response": response}}
    )

annotated_data_list[0]

{'inputs': {'text': 'Computational linguistics is an interdisciplinary field concerned with the computational modelling of natural language, as well as the study of appropriate computational approaches to linguistic questions. In general, computational linguistics draws upon linguistics, computer science, artificial intelligence, mathematics, logic, philosophy, cognitive science, cognitive psychology, psycholinguistics, anthropology and neuroscience, among others.\n\nSince the 2020s, computational linguistics has become a near-synonym of either natural language processing or language technology, with deep learning approaches, such as large language models, outperforming the specific approaches previously used in the field.\n\nOrigins\nThe field overlapped with artificial intelligence since the efforts in the United States in the 1950s to use computers to automatically translate texts from foreign languages, particularly Russian scientific journals, into English. Since rule-based approa

In [23]:
annotated_dataset_from_trace = create_dataset(
    name="annotated_dataset_from_trace",
    tags={"type": "annotated", "source": "wikipedia"},
)

In [24]:
annotated_dataset_from_trace.merge_records(annotated_data_list)